In [2]:
import requests
import pandas as pd

In [3]:
API_URL = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "query.cond": "Endometriosis",
    "pageSize": 5,
    "format": "json"
}

response = requests.get(API_URL, params=params, timeout=30)

print(response.status_code)

200


In [4]:
data = response.json()

data.keys()

dict_keys(['studies', 'nextPageToken'])

In [5]:
len(data["studies"])

5

In [6]:
data["studies"][0]

{'protocolSection': {'identificationModule': {'nctId': 'NCT06495151',
   'orgStudyIdInfo': {'id': 'E2-22-1998'},
   'organization': {'fullName': 'Ankara City Hospital Bilkent',
    'class': 'OTHER'},
   'briefTitle': 'Endometriosis and Complement System',
   'officialTitle': 'Evaluation of Serum and Peritoneal Fluid Mannose-binding Lectin Associated Serine Protease-3, Adipsin, Properdin, and Complement Factor H Levels in Endometriosis Patients'},
  'statusModule': {'statusVerifiedDate': '2024-07',
   'overallStatus': 'COMPLETED',
   'expandedAccessInfo': {'hasExpandedAccess': False},
   'startDateStruct': {'date': '2022-06-10', 'type': 'ACTUAL'},
   'primaryCompletionDateStruct': {'date': '2022-09-12', 'type': 'ACTUAL'},
   'completionDateStruct': {'date': '2022-10-07', 'type': 'ACTUAL'},
   'studyFirstSubmitDate': '2024-07-03',
   'studyFirstSubmitQcDate': '2024-07-03',
   'studyFirstPostDateStruct': {'date': '2024-07-10', 'type': 'ACTUAL'},
   'lastUpdateSubmitDate': '2024-07-11',
  

In [7]:
first_study = data["studies"][0]

protocol = first_study["protocolSection"]
protocol.keys()

dict_keys(['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'oversightModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule', 'referencesModule', 'ipdSharingStatementModule'])

In [8]:
identification = protocol["identificationModule"]

print("NCT ID:", identification["nctId"])
print("Title:", identification["briefTitle"])

NCT ID: NCT06495151
Title: Endometriosis and Complement System


In [9]:
conditions = protocol.get("conditionsModule", {}).get("conditions", [])

print(conditions)

['Endometriosis', 'Complement System', 'Alternative Complement Pathway']


In [10]:
def extract_basic_study_info(study: dict, searched_condition: str) -> dict:
    """Extract selected fields from one ClinicalTrials.gov study record."""

    protocol = study.get("protocolSection", {})

    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    design = protocol.get("designModule", {})
    conditions = protocol.get("conditionsModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    eligibility = protocol.get("eligibilityModule", {})

    enrollment_info = design.get("enrollmentInfo", {})
    lead_sponsor = sponsor.get("leadSponsor", {})

    return {
        "nct_id": identification.get("nctId"),
        "searched_condition": searched_condition,
        "brief_title": identification.get("briefTitle"),
        "listed_conditions": conditions.get("conditions", []),
        "study_type": design.get("studyType"),
        "overall_status": status.get("overallStatus"),
        "enrollment": enrollment_info.get("count"),
        "enrollment_type": enrollment_info.get("type"),
        "lead_sponsor": lead_sponsor.get("name"),
        "sponsor_class": lead_sponsor.get("class"),
        "sex": eligibility.get("sex"),
        "has_results": study.get("hasResults", False),
    }

In [11]:
records = [
    extract_basic_study_info(
        study=study,
        searched_condition="Endometriosis"
    )
    for study in data["studies"]
]

sample_df = pd.DataFrame(records)

sample_df

,nct_id,searched_condition,brief_title,listed_conditions,study_type,overall_status,enrollment,enrollment_type,lead_sponsor,sponsor_class,sex,has_results
0,NCT06495151,Endometriosis,Endometriosis and Complement System,"[Endometriosis, Complement System, Alternative...",OBSERVATIONAL,COMPLETED,58,ACTUAL,Ankara City Hospital Bilkent,OTHER,FEMALE,False
1,NCT05312528,Endometriosis,Evaluation of New Biomarkers in Stage 3 and 4 ...,[Endometriosis],OBSERVATIONAL,COMPLETED,79,ACTUAL,Bagcilar Training and Research Hospital,OTHER_GOV,FEMALE,False
2,NCT07664956,Endometriosis,Cyclobenzaprine for Postoperative Pain in Mini...,"[Minimally Invasive Surgical Procedures, Abnor...",INTERVENTIONAL,NOT_YET_RECRUITING,36,ESTIMATED,Christiana Care Health Services,OTHER,FEMALE,False
3,NCT04265781,Endometriosis,Study on the Safety of Drug BAY1817080 at Diff...,"[Endometriosis Related Pain, Overactive Bladde...",INTERVENTIONAL,COMPLETED,36,ACTUAL,Bayer,INDUSTRY,MALE,False
4,NCT07241637,Endometriosis,"Effect of Tele-Yoga on Pain, Fatigue, and Qual...","[Pelvic Pain, Endometriosis]",INTERVENTIONAL,COMPLETED,60,ACTUAL,Ankara University,OTHER,FEMALE,False


In [12]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   nct_id              5 non-null      object
 1   searched_condition  5 non-null      object
 2   brief_title         5 non-null      object
 3   listed_conditions   5 non-null      object
 4   study_type          5 non-null      object
 5   overall_status      5 non-null      object
 6   enrollment          5 non-null      int64 
 7   enrollment_type     5 non-null      object
 8   lead_sponsor        5 non-null      object
 9   sponsor_class       5 non-null      object
 10  sex                 5 non-null      object
 11  has_results         5 non-null      bool  
dtypes: bool(1), int64(1), object(10)
memory usage: 573.0+ bytes


In [13]:
sample_df[
    [
        "nct_id",
        "searched_condition",
        "study_type",
        "overall_status",
        "enrollment",
        "has_results"
    ]
]

,nct_id,searched_condition,study_type,overall_status,enrollment,has_results
0,NCT06495151,Endometriosis,OBSERVATIONAL,COMPLETED,58,False
1,NCT05312528,Endometriosis,OBSERVATIONAL,COMPLETED,79,False
2,NCT07664956,Endometriosis,INTERVENTIONAL,NOT_YET_RECRUITING,36,False
3,NCT04265781,Endometriosis,INTERVENTIONAL,COMPLETED,36,False
4,NCT07241637,Endometriosis,INTERVENTIONAL,COMPLETED,60,False


In [14]:
sample_df.to_csv(
    "../data/raw/endometriosis_sample.csv",
    index=False
)

In [15]:
data = response.json()

data.keys()

dict_keys(['studies', 'nextPageToken'])

In [16]:
len(data["studies"])

5

In [17]:
first_study = data["studies"][0]

first_study

{'protocolSection': {'identificationModule': {'nctId': 'NCT06495151',
   'orgStudyIdInfo': {'id': 'E2-22-1998'},
   'organization': {'fullName': 'Ankara City Hospital Bilkent',
    'class': 'OTHER'},
   'briefTitle': 'Endometriosis and Complement System',
   'officialTitle': 'Evaluation of Serum and Peritoneal Fluid Mannose-binding Lectin Associated Serine Protease-3, Adipsin, Properdin, and Complement Factor H Levels in Endometriosis Patients'},
  'statusModule': {'statusVerifiedDate': '2024-07',
   'overallStatus': 'COMPLETED',
   'expandedAccessInfo': {'hasExpandedAccess': False},
   'startDateStruct': {'date': '2022-06-10', 'type': 'ACTUAL'},
   'primaryCompletionDateStruct': {'date': '2022-09-12', 'type': 'ACTUAL'},
   'completionDateStruct': {'date': '2022-10-07', 'type': 'ACTUAL'},
   'studyFirstSubmitDate': '2024-07-03',
   'studyFirstSubmitQcDate': '2024-07-03',
   'studyFirstPostDateStruct': {'date': '2024-07-10', 'type': 'ACTUAL'},
   'lastUpdateSubmitDate': '2024-07-11',
  

In [18]:
protocol = first_study["protocolSection"]

protocol.keys()

dict_keys(['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'oversightModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule', 'referencesModule', 'ipdSharingStatementModule'])

In [19]:
identification = protocol["identificationModule"]

print("NCT ID:", identification["nctId"])
print("Title:", identification["briefTitle"])

NCT ID: NCT06495151
Title: Endometriosis and Complement System


In [20]:
conditions = protocol.get("conditionsModule", {}).get("conditions", [])

conditions

['Endometriosis', 'Complement System', 'Alternative Complement Pathway']

In [21]:
def extract_study_info(study, searched_condition):
    """Extract useful fields from one ClinicalTrials.gov study."""

    protocol = study.get("protocolSection", {})

    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    design = protocol.get("designModule", {})
    conditions = protocol.get("conditionsModule", {})
    sponsors = protocol.get("sponsorCollaboratorsModule", {})
    eligibility = protocol.get("eligibilityModule", {})

    enrollment_info = design.get("enrollmentInfo", {})
    lead_sponsor = sponsors.get("leadSponsor", {})

    start_date_info = status.get("startDateStruct", {})
    completion_date_info = status.get("completionDateStruct", {})

    return {
        "nct_id": identification.get("nctId"),
        "searched_condition": searched_condition,
        "brief_title": identification.get("briefTitle"),
        "official_title": identification.get("officialTitle"),
        "listed_conditions": conditions.get("conditions", []),
        "study_type": design.get("studyType"),
        "phases": design.get("phases", []),
        "overall_status": status.get("overallStatus"),
        "start_date": start_date_info.get("date"),
        "completion_date": completion_date_info.get("date"),
        "enrollment": enrollment_info.get("count"),
        "enrollment_type": enrollment_info.get("type"),
        "lead_sponsor": lead_sponsor.get("name"),
        "sponsor_class": lead_sponsor.get("class"),
        "sex": eligibility.get("sex"),
        "minimum_age": eligibility.get("minimumAge"),
        "maximum_age": eligibility.get("maximumAge"),
        "has_results": study.get("hasResults", False),
    }

In [22]:
records = [
    extract_study_info(
        study=study,
        searched_condition="Endometriosis"
    )
    for study in data["studies"]
]

sample_df = pd.DataFrame(records)

sample_df

,nct_id,searched_condition,brief_title,official_title,listed_conditions,study_type,phases,overall_status,start_date,completion_date,enrollment,enrollment_type,lead_sponsor,sponsor_class,sex,minimum_age,maximum_age,has_results
0,NCT06495151,Endometriosis,Endometriosis and Complement System,Evaluation of Serum and Peritoneal Fluid Manno...,"[Endometriosis, Complement System, Alternative...",OBSERVATIONAL,[],COMPLETED,2022-06-10,2022-10-07,58,ACTUAL,Ankara City Hospital Bilkent,OTHER,FEMALE,18 Years,44 Years,False
1,NCT05312528,Endometriosis,Evaluation of New Biomarkers in Stage 3 and 4 ...,"Diagnostic Value of Annexin V, sVCAM-1, sICAM-...",[Endometriosis],OBSERVATIONAL,[],COMPLETED,2018-02-01,2019-03-01,79,ACTUAL,Bagcilar Training and Research Hospital,OTHER_GOV,FEMALE,18 Years,50 Years,False
2,NCT07664956,Endometriosis,Cyclobenzaprine for Postoperative Pain in Mini...,Cyclobenzaprine for Postoperative Pain in Mini...,"[Minimally Invasive Surgical Procedures, Abnor...",INTERVENTIONAL,[PHASE4],NOT_YET_RECRUITING,2026-07-01,2026-12-01,36,ESTIMATED,Christiana Care Health Services,OTHER,FEMALE,18 Years,None,False
3,NCT04265781,Endometriosis,Study on the Safety of Drug BAY1817080 at Diff...,Phase 1 Dose Escalation Study to Investigate S...,"[Endometriosis Related Pain, Overactive Bladde...",INTERVENTIONAL,[PHASE1],COMPLETED,2020-02-15,2020-09-20,36,ACTUAL,Bayer,INDUSTRY,MALE,20 Years,45 Years,False
4,NCT07241637,Endometriosis,"Effect of Tele-Yoga on Pain, Fatigue, and Qual...",The Effect of Tele-Yoga Application on Chronic...,"[Pelvic Pain, Endometriosis]",INTERVENTIONAL,[NA],COMPLETED,2025-09-01,2026-06-30,60,ACTUAL,Ankara University,OTHER,FEMALE,18 Years,65 Years,False


In [23]:
sample_df[
    [
        "nct_id",
        "searched_condition",
        "study_type",
        "overall_status",
        "enrollment",
        "sponsor_class",
        "has_results",
    ]
]

,nct_id,searched_condition,study_type,overall_status,enrollment,sponsor_class,has_results
0,NCT06495151,Endometriosis,OBSERVATIONAL,COMPLETED,58,OTHER,False
1,NCT05312528,Endometriosis,OBSERVATIONAL,COMPLETED,79,OTHER_GOV,False
2,NCT07664956,Endometriosis,INTERVENTIONAL,NOT_YET_RECRUITING,36,OTHER,False
3,NCT04265781,Endometriosis,INTERVENTIONAL,COMPLETED,36,INDUSTRY,False
4,NCT07241637,Endometriosis,INTERVENTIONAL,COMPLETED,60,OTHER,False


In [24]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   nct_id              5 non-null      object
 1   searched_condition  5 non-null      object
 2   brief_title         5 non-null      object
 3   official_title      5 non-null      object
 4   listed_conditions   5 non-null      object
 5   study_type          5 non-null      object
 6   phases              5 non-null      object
 7   overall_status      5 non-null      object
 8   start_date          5 non-null      object
 9   completion_date     5 non-null      object
 10  enrollment          5 non-null      int64 
 11  enrollment_type     5 non-null      object
 12  lead_sponsor        5 non-null      object
 13  sponsor_class       5 non-null      object
 14  sex                 5 non-null      object
 15  minimum_age         5 non-null      object
 16  maximum_age         4 non-null

In [25]:
sample_df.isna().sum()

nct_id                0
searched_condition    0
brief_title           0
official_title        0
listed_conditions     0
study_type            0
phases                0
overall_status        0
start_date            0
completion_date       0
enrollment            0
enrollment_type       0
lead_sponsor          0
sponsor_class         0
sex                   0
minimum_age           0
maximum_age           1
has_results           0
dtype: int64

In [26]:
sample_df.to_csv(
    "../data/raw/endometriosis_sample.csv",
    index=False
)

In [27]:
API_URL = "https://clinicaltrials.gov/api/v2/studies"


def fetch_study_page(condition, page_size=100, page_token=None):
    """Retrieve one page of ClinicalTrials.gov studies."""

    params = {
        "query.cond": condition,
        "pageSize": page_size,
        "format": "json",
    }

    if page_token is not None:
        params["pageToken"] = page_token

    response = requests.get(
        API_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [28]:
test_page = fetch_study_page(
    condition="Endometriosis",
    page_size=10
)

len(test_page["studies"])

10

In [29]:
def fetch_all_studies(condition, page_size=100, max_pages=None):
    """Retrieve all available studies for one condition."""

    all_studies = []
    page_token = None
    page_number = 0

    while True:
        page_data = fetch_study_page(
            condition=condition,
            page_size=page_size,
            page_token=page_token
        )

        studies = page_data.get("studies", [])
        all_studies.extend(studies)

        page_number += 1

        print(
            f"{condition}: downloaded page {page_number}, "
            f"total studies = {len(all_studies)}"
        )

        page_token = page_data.get("nextPageToken")

        if not page_token:
            break

        if max_pages is not None and page_number >= max_pages:
            break

    return all_studies

In [30]:
endometriosis_test = fetch_all_studies(
    condition="Endometriosis",
    page_size=100,
    max_pages=2
)

Endometriosis: downloaded page 1, total studies = 100
Endometriosis: downloaded page 2, total studies = 200


In [31]:
len(endometriosis_test)

200

In [32]:
endometriosis_records = [
    extract_study_info(
        study=study,
        searched_condition="Endometriosis"
    )
    for study in endometriosis_test
]

endometriosis_df = pd.DataFrame(endometriosis_records)

endometriosis_df.head()

,nct_id,searched_condition,brief_title,official_title,listed_conditions,study_type,phases,overall_status,start_date,completion_date,enrollment,enrollment_type,lead_sponsor,sponsor_class,sex,minimum_age,maximum_age,has_results
0,NCT06495151,Endometriosis,Endometriosis and Complement System,Evaluation of Serum and Peritoneal Fluid Manno...,"[Endometriosis, Complement System, Alternative...",OBSERVATIONAL,[],COMPLETED,2022-06-10,2022-10-07,58.0,ACTUAL,Ankara City Hospital Bilkent,OTHER,FEMALE,18 Years,44 Years,False
1,NCT05312528,Endometriosis,Evaluation of New Biomarkers in Stage 3 and 4 ...,"Diagnostic Value of Annexin V, sVCAM-1, sICAM-...",[Endometriosis],OBSERVATIONAL,[],COMPLETED,2018-02-01,2019-03-01,79.0,ACTUAL,Bagcilar Training and Research Hospital,OTHER_GOV,FEMALE,18 Years,50 Years,False
2,NCT07664956,Endometriosis,Cyclobenzaprine for Postoperative Pain in Mini...,Cyclobenzaprine for Postoperative Pain in Mini...,"[Minimally Invasive Surgical Procedures, Abnor...",INTERVENTIONAL,[PHASE4],NOT_YET_RECRUITING,2026-07-01,2026-12-01,36.0,ESTIMATED,Christiana Care Health Services,OTHER,FEMALE,18 Years,None,False
3,NCT04265781,Endometriosis,Study on the Safety of Drug BAY1817080 at Diff...,Phase 1 Dose Escalation Study to Investigate S...,"[Endometriosis Related Pain, Overactive Bladde...",INTERVENTIONAL,[PHASE1],COMPLETED,2020-02-15,2020-09-20,36.0,ACTUAL,Bayer,INDUSTRY,MALE,20 Years,45 Years,False
4,NCT07241637,Endometriosis,"Effect of Tele-Yoga on Pain, Fatigue, and Qual...",The Effect of Tele-Yoga Application on Chronic...,"[Pelvic Pain, Endometriosis]",INTERVENTIONAL,[NA],COMPLETED,2025-09-01,2026-06-30,60.0,ACTUAL,Ankara University,OTHER,FEMALE,18 Years,65 Years,False


In [33]:
endometriosis_df.shape

(200, 18)

In [34]:
endometriosis_df["nct_id"].duplicated().sum()

np.int64(0)

In [35]:
endometriosis_df["overall_status"].value_counts()

overall_status
COMPLETED                  96
RECRUITING                 36
UNKNOWN                    34
NOT_YET_RECRUITING         14
TERMINATED                  7
ACTIVE_NOT_RECRUITING       7
WITHDRAWN                   3
ENROLLING_BY_INVITATION     2
SUSPENDED                   1
Name: count, dtype: int64

In [36]:
endometriosis_df["study_type"].value_counts()

study_type
INTERVENTIONAL    122
OBSERVATIONAL      78
Name: count, dtype: int64

In [37]:
endometriosis_df["enrollment"].describe()

count     197.000000
mean      255.284264
std       887.896666
min         0.000000
25%        43.000000
50%        86.000000
75%       180.000000
max      9979.000000
Name: enrollment, dtype: float64

In [38]:
endometriosis_df.to_csv(
    "../data/raw/endometriosis_test.csv",
    index=False
)